# NaijaBizRAG: A Retrieval-Augmented Assistant for Nigerian SME Registration & Compliance

**Author:** Samuel Yaula Dutse (SamDutse) — Lead Data Scientist, Bluehouse Technologies Ltd. | AI/Data Science Instructor, Nexus Hub Limited

**Inspired by:** an introductory RAG lesson that built a Berlin-facts tour-guide chatbot, and extended here (following the same pattern as the earlier AgriRAG Naija project) into a second real-world problem: helping first-time Nigerian entrepreneurs navigate business registration and regulatory compliance.

---

## The problem we're solving

Every year, thousands of young Nigerians try to formalize a business idea and run into the same wall: **which agency handles what, and what actually applies to a business like theirs?** CAC, FIRS, State IRS, SMEDAN, NAFDAC, SON, NITDA, CBN, PenCom, NSITF — each has a different mandate, and most first-time founders have no single, trustworthy place to check. This gap gets exploited: "runs" agents routinely overcharge for CAC registration, and businesses skip sector-specific licensing simply because they didn't know it existed, only to face fines or product seizure later.

**Our approach:** the same RAG (Retrieval-Augmented Generation) pattern used in the AgriRAG project — retrieve only from a curated set of verified regulatory facts, and generate an answer strictly grounded in that context. If a question falls outside the knowledge base, the assistant should say so rather than guess — which matters even more here than in the agriculture case, since wrong compliance advice can cost real money.

> ⚠️ **Important disclaimer, repeated throughout this notebook:** this is a teaching and portfolio project. The facts below are written for demonstration purposes and simplified for clarity. **This is not legal, tax, or regulatory advice.** Real fees, thresholds, and procedures change, and anyone relying on this for an actual business decision should verify with CAC, FIRS, SMEDAN, or a qualified professional first.

This notebook is meant to be:
1. A **teaching walkthrough** — read top to bottom to learn how RAG works, piece by piece.
2. A **portfolio project** — clone it, extend it, push it to GitHub.
3. A **starting point for students** — every section ends with a short exercise.


## What is RAG, in one paragraph?

A Large Language Model (LLM) only "knows" what was in its training data, and has no built-in awareness of a specific, curated knowledge base — in our case, a set of Nigerian SME compliance facts. **Retrieval-Augmented Generation** fixes this in two steps:

1. **Retrieve**: given a question, search a knowledge base for the most relevant pieces of text, using vector similarity rather than keyword matching.
2. **Generate**: hand those retrieved pieces of text to an LLM as *context*, and instruct it to answer using only that context.

The result is an assistant that is fluent (thanks to the LLM) but also *grounded* (thanks to retrieval) — it is constrained to the facts it was given rather than free to invent plausible-sounding but wrong compliance steps.


# 1. Setup

In [ ]:
# If you're running this in Google Colab, store your Hugging Face token as a Colab secret named HF_TOKEN.
# If you're running this locally / outside Colab, it will fall back to a manual prompt.
try:
    from google.colab import userdata
    hf_key = userdata.get('HF_TOKEN')
except ModuleNotFoundError:
    import getpass
    hf_key = getpass.getpass("Enter your Hugging Face token: ")


In [ ]:
import os
os.environ['HUGGINGFACEHUB_API_TOKEN'] = hf_key


In [ ]:
!pip install langchain-huggingface sentence-transformers langchain_community faiss-cpu -q


In [ ]:
# Import the libraries
from langchain.docstore.document import Document
from langchain.vectorstores.faiss import FAISS
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings, ChatHuggingFace, HuggingFaceEndpoint
from IPython.display import display, Markdown


# 2. The Knowledge Base

For a real deployment, this would come from verified primary sources — the CAC portal, FIRS guidelines, SMEDAN publications, and the relevant sector regulators (NAFDAC, SON, NITDA, CBN, PenCom, NSITF). For this teaching notebook, we use a **synthetic but regulation-realistic** set of facts covering company registration, tax obligations, sector-specific licensing, employment compliance, local permits, intellectual property, financing bodies, and common scam pitfalls.

Each fact is written as a short, self-contained sentence, which keeps retrieval precise since we aren't chunking large documents (a good next step — see the exercises later on).

> ⚠️ **Note:** verify every fact against a current official source before using anything like this for a real business decision.


In [ ]:
# Synthetic knowledge base: 59 facts about Nigerian SME registration & regulatory compliance
documents = [
    # --- CAC business registration ---
    "The Corporate Affairs Commission (CAC) is the federal agency responsible for registering businesses in Nigeria.",
    "A Business Name registration is the simplest structure, suited to sole proprietors and small partnerships, and does not create a separate legal entity from its owner.",
    "A Private Limited Company (Ltd) registration creates a separate legal entity, which protects the owner's personal assets from most business debts.",
    "CAC registration can be completed online through the Company Registration Portal without needing a physical visit to a CAC office.",
    "A proposed company name must be reserved and approved by CAC before full registration can proceed.",
    "A Private Limited Company in Nigeria requires a minimum of one director and one shareholder, who can be the same person.",
    "Every registered company must file Annual Returns with CAC, even in years where the business had no significant activity.",
    "Failure to file Annual Returns for multiple consecutive years can lead to a company being struck off the CAC register.",
    "Minimum share capital requirements vary by business activity; certain regulated sectors like banking require much higher minimum capital than a small retail business.",
    "A company's Memorandum and Articles of Association outline its objectives and internal governance rules and are filed with CAC at registration.",

    # --- Tax registration and obligations ---
    "The Federal Inland Revenue Service (FIRS) issues the Tax Identification Number (TIN), which is required to open most corporate bank accounts.",
    "TIN registration for a newly incorporated company can now be done automatically alongside CAC registration in many cases.",
    "Companies with an annual turnover above the VAT registration threshold are required to register for and remit Value Added Tax.",
    "Small companies below a certain turnover threshold may qualify for reduced or exempted Companies Income Tax rates under current tax law.",
    "Companies Income Tax returns are generally expected to be filed annually with FIRS, along with audited financial statements above certain turnover levels.",
    "Withholding tax is deducted at source on certain payments, such as contracts, rent, and professional services, and remitted to the relevant tax authority.",
    "State Internal Revenue Services (State IRS) handle Personal Income Tax (PAYE) for employees, which is separate from federal company taxes.",
    "Employers are required to deduct and remit Pay-As-You-Earn (PAYE) tax from employee salaries to the relevant State Internal Revenue Service.",
    "Businesses operating in more than one state may need to register with the State IRS in each state where they have a taxable presence.",
    "Late filing or non-remittance of taxes can attract penalties and interest charges under Nigerian tax law.",

    # --- SMEDAN and MSME support ---
    "The Small and Medium Enterprises Development Agency of Nigeria (SMEDAN) provides support, training, and advisory services to micro, small, and medium enterprises.",
    "SMEDAN classifies businesses into micro, small, and medium categories based on criteria such as employee count and asset value, excluding land and buildings.",
    "SMEDAN operates business development centers in various states that offer free or subsidized advisory services to entrepreneurs.",
    "Several government-backed loan schemes for MSMEs are administered in partnership with SMEDAN and participating commercial or development banks.",

    # --- Sector-specific regulatory bodies ---
    "The National Agency for Food and Drug Administration and Control (NAFDAC) regulates the registration of food, drugs, cosmetics, and related consumable products.",
    "Any business manufacturing or importing packaged food, beverages, or cosmetics for sale in Nigeria is generally required to obtain NAFDAC registration.",
    "The Standards Organisation of Nigeria (SON) certifies products for quality and safety through the SONCAP scheme, particularly for imported goods.",
    "Products requiring SONCAP certification typically need a Product Certificate before they can be cleared through Nigerian ports.",
    "The National Information Technology Development Agency (NITDA) oversees technology policy and data protection compliance for tech companies operating in Nigeria.",
    "Under the Nigeria Data Protection Act, businesses that process personal data above a certain threshold may be required to register with the Nigeria Data Protection Commission.",
    "The Central Bank of Nigeria (CBN) licenses and regulates fintech companies, including payment service providers and microfinance banks.",
    "A business dealing in imported goods generally needs to register with the Nigeria Customs Service and may require an importer's TIN-linked profile.",
    "State Ministries of Environment often require an environmental impact assessment or waste management compliance for certain manufacturing businesses.",
    "Businesses in the food and hospitality sector often require additional local government health and sanitation permits beyond federal registration.",

    # --- Employment and labor compliance ---
    "Under the Pension Reform Act, employers with three or more staff are generally required to register employees for a Contributory Pension Scheme.",
    "Employers are expected to remit a minimum combined pension contribution, split between employer and employee, to a licensed Pension Fund Administrator.",
    "The Nigeria Social Insurance Trust Fund (NSITF) provides an Employee Compensation Scheme that employers are required to contribute toward for workplace injury coverage.",
    "The Industrial Training Fund (ITF) requires employers with a certain number of staff or turnover to contribute toward staff training and development.",
    "An employment contract or letter of engagement, while not always mandatory in writing for every arrangement, is strongly recommended to avoid labor disputes.",
    "The National Minimum Wage Act sets a floor for monthly wages that employers are legally required to meet.",
    "Employers are generally required to register with the National Health Insurance Authority for applicable staff health coverage where mandated.",

    # --- Business premises and local compliance ---
    "A Business Premises Registration Permit is often required from the relevant state government before commencing physical business operations.",
    "Signage and advertisement permits from state authorities are frequently required for shop signs, billboards, and street-facing advertisements.",
    "Local Government Areas may require a separate operating permit or levy payment for businesses operating within their jurisdiction.",
    "Fire safety certification from the state fire service may be required for certain categories of commercial premises, especially those with high foot traffic.",

    # --- Intellectual property ---
    "Trademark registration in Nigeria is handled by the Trademarks, Patents and Designs Registry under the Federal Ministry of Industry, Trade and Investment.",
    "A trademark search is typically conducted before filing to check whether a similar mark is already registered in the same class of goods or services.",
    "Copyright protection in Nigeria arises automatically upon creation of an original work, but registration with the Nigerian Copyright Commission can strengthen enforcement.",
    "Patent applications in Nigeria are examined for novelty and are also filed through the Trademarks, Patents and Designs Registry.",

    # --- Financing and grants ---
    "The Bank of Industry (BOI) offers development financing for manufacturing, agro-processing, and other productive sector businesses.",
    "The Development Bank of Nigeria (DBN) provides wholesale funding to participating financial institutions for on-lending to MSMEs.",
    "Grant and competition programs run periodically by government agencies and private organizations can provide non-repayable funding to qualifying startups.",
    "Some state governments run their own SME support funds or grant schemes distinct from federal programs.",

    # --- Common pitfalls and scams ---
    "Business owners are advised to verify agents claiming to offer 'express' CAC registration, as some unregistered intermediaries overcharge or misrepresent the process.",
    "CAC registration fees and timelines are published on the official CAC portal, and prices significantly above the published rate should raise questions.",
    "Using a business's official TIN and CAC registration number helps verify the legitimacy of invoices, contracts, and formal business dealings.",
    "Operating a business without required sector-specific licenses, such as NAFDAC or SON certification, can result in product seizure, fines, or business closure.",
    "Keeping separate business and personal bank accounts is recommended for financial clarity and is often required to open a proper corporate account.",
    "Engaging a qualified accountant or company secretary, even part-time, can help a small business stay compliant with filing deadlines it might otherwise miss.",
]


In [ ]:
# Check how many documents we have
print(f"We have {len(documents)} documents")


### 🧪 Exercise 1
Add 5–10 more facts of your own — perhaps about a state-specific permit, an industry you're familiar with (agro-processing, fashion, tech), or a compliance step you've personally had to navigate while reviving DigitALL Solution Limited. Keep each fact to one clear sentence, the same way the examples above are written.


# 3. Embeddings — Turning Text Into Vectors

An embedding model converts each sentence into a vector of numbers such that sentences with similar *meaning* end up close together in vector space, even without shared keywords — so "how do I stop paying too much for CAC registration" can still retrieve the fact about verifying agents and published fees.

We use `sentence-transformers/all-MiniLM-L6-v2`, a small, fast, widely-used embedding model — good enough for a teaching notebook and light enough to run on a free Colab CPU.


In [ ]:
# Wrap each string in a Document object (LangChain's standard text container)
docs = [Document(page_content=text) for text in documents]


In [ ]:
# Use a transformer-based embedding model
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [ ]:
# Create a FAISS vector store from the documents
# FAISS (Facebook AI Similarity Search) indexes the vectors so we can search them quickly
faiss_store = FAISS.from_documents(docs, embedding_model)


In [ ]:
index = faiss_store.index

# Print total number of indexed vectors
print(f"Total number of indexed vectors: {index.ntotal}")

# Print total number of dimensions per vector
print(f"Total number of dimensions: {index.d}")

# Print the embedding for the first document, just to see what one actually looks like
print(f"Embedding for the first document:\n{index.reconstruct(0)}")


# 4. Retrieval System — Finding the Right Facts

Given a founder's question, we embed the question the same way we embedded the documents, then find the `k` documents whose vectors are closest to it. This is a **similarity search**, not a keyword search — so "I want to sell packaged snacks, what do I need?" can retrieve the NAFDAC fact even without the word "NAFDAC" appearing in the question.


In [ ]:
# Try a raw similarity search
query = "I want to start a small food packaging business, what registrations do I need?"
k = 5
retrieved_docs = faiss_store.similarity_search(query, k)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"{i}. {doc.page_content}")


In [ ]:
# Wrap retrieval in a reusable function
def get_relevant_documents(query, k=5):
    return faiss_store.similarity_search(query, k)


### 🧪 Exercise 2
Try `faiss_store.similarity_search_with_score(query, k)` instead of `similarity_search`. Compare the scores for a well-covered question (e.g. about CAC) against a question near the edges of the knowledge base (e.g. about international trademark filing). What does that tell you about setting a relevance threshold later?


# 5. Generative System — Turning Retrieved Facts Into an Answer

Now we hand the retrieved facts to an LLM as context, along with a system prompt that constrains it to only use that context. Because this domain is higher-stakes than tourist trivia or farming tips, the persona here leans calm and precise rather than casual, and the system prompt explicitly instructs the model to flag that this isn't professional advice.

We use `microsoft/Phi-3.5-mini-instruct` via the Hugging Face Inference Endpoint, the same lightweight instruction-tuned model used in the earlier lessons.


In [ ]:
# Load the LLM
llm = HuggingFaceEndpoint(
    repo_id="microsoft/Phi-3.5-mini-instruct",
    task="text-generation"
)
chat_model = ChatHuggingFace(llm=llm)


In [ ]:
# Define the system and human messages
def generative_system(query, context):
    messages = [
        SystemMessage(content=f"""
        You are a calm, precise Nigerian business registration and compliance advisor,
        helping first-time entrepreneurs understand what applies to their business.
        Only answer using information from {context}.
        If the context does not contain the answer, say you don't have that information yet
        and suggest the founder check the official CAC, FIRS, or SMEDAN channels, or consult a professional.
        Always remind the user this is general guidance, not formal legal or tax advice."""),
        HumanMessage(content=f"Answer this founder's question: {query}, based on this context: {context}")
    ]
    ai_output = chat_model.invoke(messages)
    return display(Markdown(ai_output.content))


### 🧪 Exercise 3
Try removing the "this is not legal or tax advice" instruction from the system prompt and re-run a few queries. Notice how the tone changes. This is a good discussion point with students: **prompt design isn't just about accuracy, it's about appropriate framing for the stakes of the domain.**


# 6. Combining Retrieval and Generation = RAG

Same five-line pattern as before: retrieve the most relevant facts, then generate a grounded answer from them.


In [ ]:
# Build the RAG system
def rag(query):
    context = get_relevant_documents(query)
    return generative_system(query, context)


In [ ]:
# Test the RAG system with a question our knowledge base can answer
query = "I'm about to hire my third staff member, is there anything I need to register for them?"
rag(query)


In [ ]:
# Prepare test queries, including one the knowledge base genuinely cannot answer
query_list = [
    "What's the difference between registering a Business Name and a Limited Company?",
    "Do I need any special registration to sell homemade snacks?",
    "What is the exact CAC registration fee for a Limited Company this year?",  # not reliably answerable — fees change and aren't in our static context
]


In [ ]:
# Test the RAG system across all queries
for query in query_list:
    print(f"Q: {query}")
    rag(query)
    print("\n---\n")


Notice the last question, about exact current fees. Our knowledge base deliberately does not hardcode specific fee amounts, since those change and a stale number is worse than no number. A well-grounded RAG system should point the founder to the official CAC portal rather than inventing a figure. If your assistant states a confident number here, that's a sign the system prompt needs tightening, or that fee-type facts need to be excluded from the knowledge base entirely and handled by a live lookup instead.


# 7. What's Actually Happening Under the Hood

```
Founder's question
       │
       ▼
 Embed the question  ──────────────►  vector
       │
       ▼
 Search FAISS index for nearest k document vectors
       │
       ▼
 Retrieved facts (context)
       │
       ▼
 System prompt + context + question  ──────────►  LLM (Phi-3.5-mini)
       │
       ▼
 Grounded answer, with an explicit "not professional advice" reminder
```

Every RAG system, no matter how advanced, is a variation on this same loop: **embed → retrieve → augment the prompt → generate**. What changes between AgriRAG and NaijaBizRAG isn't the architecture — it's the knowledge base and the persona/system-prompt framing appropriate to the domain's stakes.


# 8. Exercises to Extend This Project

Pick one or more of these to make the project your own before pushing it to GitHub:

1. **Chunking real documents.** Replace the synthetic fact list with real text from the CAC, FIRS, and SMEDAN websites, split into chunks using `RecursiveCharacterTextSplitter`, and re-run the pipeline.
2. **Source citation.** Modify `generative_system` to also display which retrieved facts were used — this matters even more here than in AgriRAG, since founders may want to double-check a compliance claim.
3. **Similarity score threshold.** Use `similarity_search_with_score` and discard retrieved documents below a relevance threshold, instead of always returning `k` documents.
4. **Business-type routing.** Add a first step that asks the user what kind of business they're running (retail, food production, tech, services) and pre-filter the knowledge base accordingly.
5. **Evaluation.** Write 15–20 test questions with known correct answers, and manually score how often the RAG system is accurate vs. how often it correctly says "I don't know" or redirects to an official source.
6. **Deployment.** Wrap `rag()` in a Gradio interface and deploy it to Hugging Face Spaces — a natural resource to share with members of your own hub.
7. **Freshness handling.** Add a mechanism to flag any fact involving a fee or numeric threshold as "verify current amount," since these are the facts most likely to go stale.

## Limitations to be upfront about

- The knowledge base here is **synthetic** and simplified for teaching — it is **not legal, tax, or regulatory advice**.
- Fees, thresholds, and specific procedures referenced in Nigerian regulation change over time; none of that should be treated as current without verification.
- Small instruction-tuned models can still occasionally drift from the provided context; always test with adversarial or out-of-scope questions, especially ones asking for exact numbers.
- A real deployment would need review by a qualified accountant, company secretary, or legal practitioner before founders rely on it for actual decisions.

---

*This notebook was built as a teaching and portfolio project by Samuel Yaula Dutse, applying the same retrieval-augmented generation pattern used in the AgriRAG Naija project to a second real-world problem: helping Nigerian entrepreneurs navigate business registration and compliance.*
